# schema

> The audio-transcript layer schema (where-graph-begins locked layer schema): `Source -> AudioSegment(coarse, owns the model-input WAV) -> Transcript(per-transcriber variants)` emitted by transcription, extended with the fine `Segment` spine by decomposition. Deterministic identity tuples per the stage-5 ratified rule; `Document` from the pre-CR-18 era dissolves into `Source`.

In [ ]:
#| default_exp schema

In [ ]:
#| export
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from cjm_context_graph_layer.identity import derive_node_id
from cjm_context_graph_layer.grammar import OverlayRelations, attribution, make_edge
from cjm_context_graph_primitives.locators import FileRef, GraphNodeRef
from cjm_context_graph_primitives.slices import TimeSlice, CharSlice, FullContent
from cjm_context_graph_primitives.provenance import SourceRef

In [ ]:
#| export
class TranscriptGraphLabels:
    """Node labels of the audio-transcript layer schema."""
    SOURCE = "Source"               # Provenance root: one ingested media file
    AUDIO_SEGMENT = "AudioSegment"  # Coarse ~5-min spine member; owns the model-input WAV
    TRANSCRIPT = "Transcript"       # Per-transcriber text variant of one AudioSegment
    SEGMENT = "Segment"             # Fine spine member: one VAD chunk (immutable audio + correctable text)

    @classmethod
    def all(cls) -> list:  # All schema labels
        """All schema labels."""
        return [cls.SOURCE, cls.AUDIO_SEGMENT, cls.TRANSCRIPT, cls.SEGMENT]

In [ ]:
#| export
def source_node_id(
    content_hash: str,  # Content hash of the source media file ("algo:hexdigest")
) -> str:  # Deterministic Source node id
    """Source identity = the ingested file's content hash (Thread-1 ingested-root identity)."""
    return derive_node_id("source", content_hash)

In [ ]:
#| export
def audio_segment_node_id(
    source_id: str,  # Owning Source node id
    start: float,    # Boundary start (source-coordinate seconds)
    end: float,      # Boundary end (source-coordinate seconds)
) -> str:  # Deterministic AudioSegment node id
    """AudioSegment identity = (source, boundary range).

    Conversion-config-independent: the model-input WAV is payload/provenance,
    not identity — the boundary computation is pure and deterministic, so
    re-derivation reproduces the id."""
    return derive_node_id("audio-segment", source_id, start, end)

In [ ]:
#| export
def transcript_node_id(
    audio_segment_id: str,  # Owning AudioSegment node id
    transcriber: str,       # Transcriber capability name (e.g. "cjm-transcription-plugin-whisper")
    config_hash: str,       # Transcriber config hash
) -> str:  # Deterministic Transcript node id
    """Transcript identity = (audio segment, transcriber, config) — MIRRORS the
    capability cache key UNIQUE(audio_path, config_hash), so the graph node is
    the durable face of the cached row (the structural E13 fix)."""
    return derive_node_id("transcript", audio_segment_id, transcriber, config_hash)

In [ ]:
#| export
def segment_node_id(
    audio_segment_id: str,  # Owning AudioSegment node id
    vad_config_hash: str,   # VAD capability config hash (skeleton identity input)
    chunk_start: float,     # VAD chunk start (chunk-local seconds within the AudioSegment)
    chunk_end: float,       # VAD chunk end (chunk-local seconds)
) -> str:  # Deterministic Segment node id
    """Fine Segment identity = audio-side only (audio segment, VAD config, chunk
    range) — so the skeleton's ids are SHARED across transcribers by
    construction (C4 "store agreement once" falls out of identity design)."""
    return derive_node_id("segment", audio_segment_id, vad_config_hash, chunk_start, chunk_end)

In [ ]:
#| export
@dataclass
class SourceNode:
    """The provenance root: one ingested media file."""
    content_hash: str            # Content hash over the file ("algo:hexdigest"; the identity input)
    path: str                    # Original media path (location, may dangle; identity is the hash)
    title: Optional[str] = None  # Display title; None = path stem
    media_type: str = "audio"    # Media type

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id."""
        return source_node_id(self.content_hash)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the Source node wire dict (root_kind=ingested; FileRef provenance)."""
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.SOURCE,
            "properties": {
                "title": self.title or Path(self.path).stem,
                "media_type": self.media_type,
                "path": self.path,
                "root_kind": "ingested",
            },
            "sources": [SourceRef(locator=FileRef(path=self.path),
                                  content_hash=self.content_hash).to_dict()],
        }

In [ ]:
#| export
@dataclass
class AudioSegmentNode:
    """Coarse ~5-min spine member. Owns the model-input WAV (E14: the audio of
    record) as payload/provenance — the WAV is NOT its own node.

    Provenance note (deliberate): the SourceRef anchors the owned model-input
    WAV artifact (hash_file-verifiable). A hash over the sliced ORIGINAL source
    bytes is not materializable without extracting per-range artifacts (decoded
    ranges are not byte ranges); the structural chain to the Source rides the
    PART_OF edge + the start/end properties."""
    source: str                       # Owning Source node id
    index: int                        # Position in the coarse spine (0-based)
    start: float                      # Boundary start (source-coordinate seconds)
    end: float                        # Boundary end (source-coordinate seconds)
    model_input_path: str             # The 16kHz mono WAV consumed by transcription/VAD/FA
    model_input_hash: str             # Content hash over that WAV
    segment_path: Optional[str] = None  # Source-codec cut file (archival pointer)

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id."""
        return audio_segment_node_id(self.source, self.start, self.end)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the AudioSegment node wire dict."""
        props: Dict[str, Any] = {
            "index": self.index,
            "start": self.start,
            "end": self.end,
            "model_input_path": self.model_input_path,
            "source_id": self.source,
        }
        if self.segment_path:
            props["segment_path"] = self.segment_path
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.AUDIO_SEGMENT,
            "properties": props,
            "sources": [SourceRef(locator=FileRef(path=self.model_input_path),
                                  content_hash=self.model_input_hash).to_dict()],
        }

In [ ]:
#| export
@dataclass
class TranscriptNode:
    """One transcriber's text for one AudioSegment (per-transcriber variants at
    the coarse layer — cross-transcriber divergence lives here, C4/C14)."""
    audio_segment: str            # Owning AudioSegment node id
    transcriber: str              # Transcriber capability name
    config_hash: str              # Transcriber config hash
    text: str                     # The transcript text (stored ONCE here; fine Segments slice into it)
    audio_hash: str               # Content hash of the consumed model-input WAV
    metadata: Dict[str, Any] = field(default_factory=dict)  # Transcriber-reported metadata
    asserted_at: Optional[float] = None  # Derivation timestamp; None = now

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id."""
        return transcript_node_id(self.audio_segment, self.transcriber, self.config_hash)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the Transcript node wire dict (capability attribution included)."""
        props: Dict[str, Any] = {
            "transcriber": self.transcriber,
            "config_hash": self.config_hash,
            "text": self.text,
            "audio_segment_id": self.audio_segment,
        }
        if self.metadata:
            props["metadata"] = dict(self.metadata)
        props.update(attribution(f"capability:{self.transcriber}", method="transcribe",
                                 asserted_at=self.asserted_at))
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.TRANSCRIPT,
            "properties": props,
            "sources": [SourceRef(locator=GraphNodeRef(node_id=self.audio_segment),
                                  content_hash=self.audio_hash,
                                  slice=FullContent("audio")).to_dict()],
        }

    def derived_edge(self) -> Dict[str, Any]:  # Edge wire dict
        """DERIVED_FROM edge: this Transcript derives from its AudioSegment."""
        return make_edge(self.id, self.audio_segment, OverlayRelations.DERIVED_FROM)

In [ ]:
#| export
@dataclass
class TranscriptSliceRef:
    """One per-transcriber char-range reference for a fine Segment: where this
    segment's text lives inside a Transcript node's text (Thread-1
    slices-until-promoted — variants are slices, never duplicated fine text)."""
    transcript: str  # Transcript node id
    start_char: int  # Slice start into the Transcript's text
    end_char: int    # Slice end
    text: str        # The sliced text (content-hashed for the ref)

    def to_source_ref(self) -> Dict[str, Any]:  # SourceRef wire dict
        """Build the slice-shaped provenance ref."""
        return SourceRef(locator=GraphNodeRef(node_id=self.transcript),
                         content_hash=SourceRef.compute_hash(self.text.encode()),
                         slice=CharSlice(start=self.start_char, end=self.end_char)).to_dict()

In [ ]:
#| export
@dataclass
class SegmentNode:
    """Fine spine member: one VAD chunk — IMMUTABLE audio range + CORRECTABLE
    text (the immutable-audio/mutable-text spine seam).

    Layer-0 `text` is the ACCURACY transcriber's alignment; the designation is
    per-segment provenance, not global config (`text_from` names the
    authoritative Transcript; every transcriber's char range rides
    `text_slices`, the authoritative one included)."""
    audio_segment: str        # Owning AudioSegment node id
    vad_config_hash: str      # VAD config hash (skeleton identity input)
    chunk_start: float        # VAD chunk start (chunk-local seconds within the AudioSegment WAV)
    chunk_end: float          # VAD chunk end (chunk-local seconds)
    index: int                # Source-wide fine-spine index (0-based)
    start_time: float         # Source-coordinate start (navigation)
    end_time: float           # Source-coordinate end
    text: str = ""            # Layer-0 text (accuracy alignment; "" = no aligned words, D14 class)
    audio_hash: str = ""      # Content hash of the owning AudioSegment's model-input WAV
    source: Optional[str] = None     # Source node id (convenience for direct filters)
    text_from: Optional[str] = None  # Authoritative Transcript node id (None when text is empty)
    text_slices: List[TranscriptSliceRef] = field(default_factory=list)  # All per-transcriber slice refs

    @property
    def id(self) -> str:  # Deterministic node id
        """Deterministic node id (audio-side identity; shared across transcribers)."""
        return segment_node_id(self.audio_segment, self.vad_config_hash,
                               self.chunk_start, self.chunk_end)

    def to_graph_node(self) -> Dict[str, Any]:  # Node wire dict
        """Build the Segment node wire dict (audio ref + per-transcriber text slice refs)."""
        props: Dict[str, Any] = {
            "text": self.text,
            "index": self.index,
            "start_time": self.start_time,
            "end_time": self.end_time,
            "chunk_start": self.chunk_start,
            "chunk_end": self.chunk_end,
            "audio_segment_id": self.audio_segment,
        }
        if self.source:
            props["source_id"] = self.source
        if self.text_from:
            props["text_from"] = self.text_from
        sources = [SourceRef(locator=GraphNodeRef(node_id=self.audio_segment),
                             content_hash=self.audio_hash,
                             slice=TimeSlice(start=self.chunk_start, end=self.chunk_end)).to_dict()]
        sources.extend(ts.to_source_ref() for ts in self.text_slices)
        return {
            "id": self.id,
            "label": TranscriptGraphLabels.SEGMENT,
            "properties": props,
            "sources": sources,
        }

In [ ]:
# tests — identity determinism + cross-type distinctness
sid = source_node_id("sha256:abc")
assert sid == source_node_id("sha256:abc")
a1 = audio_segment_node_id(sid, 0.0, 300.2)
assert a1 == audio_segment_node_id(sid, 0.0, 300.2)
assert a1 != audio_segment_node_id(sid, 0.0, 300.3)
t1 = transcript_node_id(a1, "whisper", "cfg1")
assert t1 != transcript_node_id(a1, "voxtral", "cfg1")
assert t1 != transcript_node_id(a1, "whisper", "cfg2")
s1 = segment_node_id(a1, "vadcfg", 1.5, 4.25)
assert s1 == segment_node_id(a1, "vadcfg", 1.5, 4.25)
ids = {sid, a1, t1, s1}
assert len(ids) == 4, "kinds never collide"
print("identity tests OK")

identity tests OK


In [ ]:
# tests — Source / AudioSegment wire shapes
src = SourceNode(content_hash="sha256:abc", path="/media/ep1.mp3")
n = src.to_graph_node()
assert n["label"] == "Source" and n["id"] == source_node_id("sha256:abc")
assert n["properties"]["title"] == "ep1" and n["properties"]["root_kind"] == "ingested"
assert n["sources"][0]["content_hash"] == "sha256:abc"
assert SourceNode(content_hash="sha256:abc", path="/x.mp3", title="Custom").to_graph_node()["properties"]["title"] == "Custom"

aseg = AudioSegmentNode(source=src.id, index=0, start=0.0, end=300.2,
                        model_input_path="/cache/seg0.wav", model_input_hash="sha256:wav0",
                        segment_path="/cuts/seg0.mp3")
an = aseg.to_graph_node()
assert an["id"] == audio_segment_node_id(src.id, 0.0, 300.2)
assert an["label"] == "AudioSegment"
assert an["properties"]["source_id"] == src.id and an["properties"]["segment_path"] == "/cuts/seg0.mp3"
assert an["sources"][0]["content_hash"] == "sha256:wav0"
print("source/audio-segment shape tests OK")

source/audio-segment shape tests OK


In [ ]:
# tests — Transcript shape + attribution + derived edge
t = TranscriptNode(audio_segment=a1, transcriber="whisper", config_hash="cfg1",
                   text="hello world", audio_hash="sha256:wav0",
                   metadata={"model": "base"}, asserted_at=123.0)
tn = t.to_graph_node()
assert tn["id"] == transcript_node_id(a1, "whisper", "cfg1")
assert tn["properties"]["actor"] == "capability:whisper"
assert tn["properties"]["method"] == "transcribe" and tn["properties"]["asserted_at"] == 123.0
assert tn["properties"]["text"] == "hello world" and tn["properties"]["metadata"] == {"model": "base"}
assert tn["sources"][0]["slice"]["kind"] == "full"
e = t.derived_edge()
assert e["relation_type"] == "DERIVED_FROM" and e["source_id"] == tn["id"] and e["target_id"] == a1
assert e["id"] == t.derived_edge()["id"], "deterministic edge id"
print("transcript shape tests OK")

transcript shape tests OK


In [ ]:
# tests — Segment shape: audio ref + authoritative/variant text slices; empty segment
acc = TranscriptSliceRef(transcript="t-acc", start_char=0, end_char=11, text="hello world")
var = TranscriptSliceRef(transcript="t-lw", start_char=0, end_char=10, text="helo world")
seg = SegmentNode(audio_segment=a1, vad_config_hash="vadcfg", chunk_start=1.5, chunk_end=4.25,
                  index=7, start_time=101.5, end_time=104.25, text="hello world",
                  audio_hash="sha256:wav0", source=sid, text_from="t-acc",
                  text_slices=[acc, var])
sn = seg.to_graph_node()
assert sn["id"] == segment_node_id(a1, "vadcfg", 1.5, 4.25)
assert sn["properties"]["text"] == "hello world" and sn["properties"]["text_from"] == "t-acc"
assert sn["properties"]["source_id"] == sid
assert len(sn["sources"]) == 3
audio_ref, acc_ref, var_ref = sn["sources"]
assert audio_ref["slice"]["kind"] == "time" and audio_ref["slice"]["start"] == 1.5
assert acc_ref["slice"]["kind"] == "char" and acc_ref["content_hash"] == SourceRef.compute_hash(b"hello world")
assert var_ref["content_hash"] == SourceRef.compute_hash(b"helo world")
# identity is audio-side: text changes do NOT change the id (skeleton shared across transcribers)
seg2 = SegmentNode(audio_segment=a1, vad_config_hash="vadcfg", chunk_start=1.5, chunk_end=4.25,
                   index=7, start_time=101.5, end_time=104.25, text="different text")
assert seg2.id == seg.id
# empty (D14-class) segment: audio ref only, no text_from
empty = SegmentNode(audio_segment=a1, vad_config_hash="vadcfg", chunk_start=9.0, chunk_end=9.5,
                    index=8, start_time=109.0, end_time=109.5, audio_hash="sha256:wav0")
en = empty.to_graph_node()
assert en["properties"]["text"] == "" and "text_from" not in en["properties"]
assert len(en["sources"]) == 1
print("segment shape tests OK")

segment shape tests OK
